# Automated ADF Sensitivity Workflow on Bank Time Series
This notebook implements an automated econometric testing workflow. It evaluates the sensitivity of Augmented Dickey-Fuller (ADF) stationarity test results across multiple regression types (`c` as default, and `n` for no constant/trend) and lag selection methods (`AIC`, `BIC`, and `t-stat`).

In [3]:
import numpy as np
import pandas as pd
import statsmodels.tsa.stattools as ts

# 1. Setup Placeholders and Inputs
# [DATA_SOURCE]
data_source_path = '/content/drive/MyDrive/Deposit Beta/consolidated_fdic_quarterly_financials_2011-2026_merger_adjusted.csv'

# [BANK_LIST] - Specified flagged banks
banks = [
    'Glacier Bank',
    'Huntington National Bank',
    'Manufacturers and Traders Trust Company'
]

# [TIME_SERIES_DATA]
# Load the real dataset
df_raw = pd.read_csv(data_source_path)

# Assuming the CSV has columns like 'bank_name' (or similar) and 'deposit_rate'.
# Let's clean the bank name column casing/spacing if needed, and map the series.
data_per_bank = {}

# Identify the exact column names
# We will locate the bank name column dynamically (e.g. 'name', 'bank_name', 'institution')
bank_col = None
for col in df_raw.columns:
    if col.lower() in ['bank_name', 'name', 'institution_name', 'bank', 'institution']:
        bank_col = col
        break

if bank_col is None:
    # Fallback to the first string column or default to 'bank_name'
    bank_col = 'bank_name'

for bank in banks:
    # Filter data for this specific bank and extract 'deposit_rate'
    bank_data = df_raw[df_raw[bank_col].str.lower() == bank.lower()]

    # Ensure data is sorted chronologically if there is a date column
    date_col = None
    for col in bank_data.columns:
        if 'date' in col.lower() or 'quarter' in col.lower():
            date_col = col
            break
    if date_col:
        bank_data = bank_data.sort_values(by=date_col)

    # Extract deposit_rate, drop missing values to ensure ADF can execute properly
    series_data = bank_data['deposit_rate'].dropna().values #switch to "deposit_rate" if using pre-M&A adjusted source, otherwise 'rate_pf_clean'
    data_per_bank[bank] = series_data

# [OUTPUT_FORMAT]
output_csv_path = 'adf_sensitivity_results.csv'

In [4]:
results_list = []

# 2. Iterate over banks and run ADF Tests
for bank_name in banks:
    series = data_per_bank[bank_name]
    diff_series = np.diff(series)

    # Skip if no data was found or if length is insufficient
    if len(series) == 0:
        print(f"Warning: No data found for {bank_name}.")
        continue

    # Compare default regression 'c' (constant) with 'n' (no constant, no trend)
    for regression_type in ['c', 'n']:

        # Test with lag selection in this order: AIC, BIC, and t-stat ('t-stat' mapped to 't-stat' in statsmodels)
        for lag_selection_type in ['AIC', 'BIC', 't-stat']:
            autolag = lag_selection_type.lower()

            try:
                # Run the ADF test using statsmodels
                adf_result = ts.adfuller(
                    diff_series,
                    regression=regression_type,
                    autolag=autolag,
                    result_object = True
                )

                q_stat = adf_result[0]
                p_value = adf_result[1]

            except Exception as e:
                # Handle exceptions (e.g., if a test fails due to series constraints or length)
                q_stat = np.nan
                p_value = np.nan

            results_list.append({
                'bank_name': f"{bank_name} I(1)",
                'regression_type': regression_type,
                'lag_selection_type': lag_selection_type,
                'Q_stat': q_stat,
                'p_value': p_value
            })

# 3. Create DataFrame
df_results = pd.DataFrame(results_list)

# 4. Order the final data frame by bank_name, then lag_selection_type, then regression_type
df_results = df_results.sort_values(by=['bank_name', 'lag_selection_type', 'regression_type']).reset_index(drop=True)

# Save to the specified output format
df_results.to_csv(output_csv_path, index=False)
print(f"Results successfully written to {output_csv_path}")

# Display the sorted results DataFrame
display(df_results)

Results successfully written to adf_sensitivity_results.csv


,bank_name,regression_type,lag_selection_type,Q_stat,p_value
0,Glacier Bank I(1),c,AIC,-2.639751,8.505659e-02
1,Glacier Bank I(1),n,AIC,-2.760372,5.625891e-03
2,Glacier Bank I(1),c,BIC,-9.605240,1.873819e-16
3,Glacier Bank I(1),n,BIC,-9.679696,5.102918e-17
4,Glacier Bank I(1),c,t-stat,-3.558846,6.598327e-03
5,Glacier Bank I(1),n,t-stat,-3.584903,3.554124e-04
6,Huntington National Bank I(1),c,AIC,-1.966777,3.013276e-01
7,Huntington National Bank I(1),n,AIC,-1.577809,1.079110e-01
8,Huntington National Bank I(1),c,BIC,-3.978251,1.528774e-03
9,Huntington National Bank I(1),n,BIC,-3.960377,8.561507e-05


In [7]:
import warnings
# Suppress Interpolation Warnings and FutureWarnings for clean presentation
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

kpss_results_list = []
output_kpss_csv_path = 'kpss_sensitivity_results.csv'

# 2. Iterate over banks and run KPSS Tests
for bank_name in banks:
    series = data_per_bank[bank_name]
    diff_series = np.diff(series)
    n_obs = len(diff_series)

    # Skip if no data was found or if length is insufficient
    if n_obs == 0:
        print(f"Warning: No data found for {bank_name}.")
        continue

    # Calculate explicit values for 'long' and 'short' lags to avoid NaN failures
    # statsmodels legacy-auto/auto sets lags dynamically, while 'short'/'long' use standard rules.
    lags_dict = {
        'auto': 'auto',
        'long': int(np.ceil(12 * (n_obs / 100) ** 0.25)),
        'short': int(np.ceil(4 * (n_obs / 100) ** 0.25))
    }

    # KPSS regression types: 'c' (constant) and 'ct' (constant and trend)
    for regression_type in ['c', 'ct']:

        # Let's check sensitivity across auto-lag selection or specified lags
        for lag_selection_type in ['auto', 'long', 'short']:
            nlags_val = lags_dict[lag_selection_type]

            try:
                # Run the KPSS test using statsmodels with result_object=False for backwards compatibility
                kpss_result = ts.kpss(
                    series,
                    regression=regression_type,
                    nlags=nlags_val,
                    store=False
                )

                q_stat = kpss_result[0]
                p_value = kpss_result[1]

            except Exception as e:
                # Handle exceptions gracefully
                q_stat = np.nan
                p_value = np.nan

            kpss_results_list.append({
                'bank_name': f"{bank_name} I(0)",
                'regression_type': regression_type,
                'lag_selection_type': lag_selection_type,
                'Q_stat': q_stat,
                'p_value': p_value
            })

# 3. Create DataFrame
df_kpss_results = pd.DataFrame(kpss_results_list)

# 4. Order the final data frame by bank_name, then lag_selection_type, then regression_type
df_kpss_results = df_kpss_results.sort_values(by=['bank_name', 'lag_selection_type', 'regression_type']).reset_index(drop=True)

# Save to the specified output format
df_kpss_results.to_csv(output_kpss_csv_path, index=False)
print(f"Results successfully written to {output_kpss_csv_path}")

# Display the sorted results DataFrame
display(df_kpss_results)

Results successfully written to kpss_sensitivity_results.csv


,bank_name,regression_type,lag_selection_type,Q_stat,p_value
0,Glacier Bank I(0),c,auto,0.465840,0.049360
1,Glacier Bank I(0),ct,auto,0.242657,0.010000
2,Glacier Bank I(0),c,long,0.264913,0.100000
3,Glacier Bank I(0),ct,long,0.146123,0.049898
4,Glacier Bank I(0),c,short,0.465840,0.049360
5,Glacier Bank I(0),ct,short,0.242657,0.010000
6,Huntington National Bank I(0),c,auto,0.585036,0.023997
7,Huntington National Bank I(0),ct,auto,0.193670,0.018374
8,Huntington National Bank I(0),c,long,0.382595,0.084657
9,Huntington National Bank I(0),ct,long,0.130534,0.078641
